# Improved Prompt Engineering with Gemini

This notebook demonstrates how to apply advanced prompt engineering techniques to the evaluation system built in the previous notebook. We will improve the system's reliability and performance by:

1.  **Using XML Tags:** Structuring inputs for better parsing and clarity.
2.  **Few-Shot Prompting:** Providing high-quality examples to guide the model.
3.  **Specific & Direct Instructions:** Reducing ambiguity in task descriptions.
4.  **Chain of Thought (Implicit):** Structuring output schemas to encourage reasoning.

## 1. Setup

Identical setup to the original notebook, using `gemini-2.5-flash` for speed and reliability.

Gemini calls in this notebook now use a shared Tenacity-based retry helper for `429` and `RESOURCE_EXHAUSTED` responses.

In [36]:
import os
import json
import concurrent.futures
import sys
from pathlib import Path
from textwrap import dedent
from statistics import mean
from dotenv import load_dotenv
from google import genai
from google.genai import types
from IPython.display import display, Markdown, HTML
from pydantic import BaseModel
from typing import List, Dict

project_root = next(
    (path for path in [Path.cwd(), *Path.cwd().parents] if (path / "requirements.txt").exists()),
    Path.cwd(),
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from utils.gemini_retry import generate_content_with_retry

load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY")
client = genai.Client(api_key=api_key)
model = "gemini-3-flash-preview"

def chat(messages, system=None, temperature=1.0, stop_sequences=[], response_mime_type=None, max_retries=5):
    contents = []
    for m in messages:
        role = "user" if m["role"] == "user" else "model"
        contents.append({"role": role, "parts": [{"text": m["content"]}]})
    
    config = types.GenerateContentConfig(
        temperature=temperature,
        max_output_tokens=2048,
        stop_sequences=stop_sequences,
        system_instruction=system,
        response_mime_type=response_mime_type
    )
    
    response = generate_content_with_retry(
        client=client,
        model=model,
        contents=contents,
        config=config,
        max_attempts=max_retries,
    )
    return response.text

## 2. Helper Function: HTML Report Generator

This function (from the original notebook) converts the raw JSON evaluation results into a clean, styled HTML report.

In [37]:
def generate_prompt_evaluation_report(evaluation_results):
    total_tests = len(evaluation_results)
    scores = [result["score"] for result in evaluation_results]
    avg_score = mean(scores) if scores else 0
    max_possible_score = 10
    pass_rate = (
        100 * len([s for s in scores if s >= 7]) / total_tests if total_tests else 0
    )

    html = f"""
    <!DOCTYPE html>
    <html lang="en">
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1.0">
        <title>Prompt Evaluation Report</title>
        <style>
            body {{ font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif; line-height: 1.6; padding: 20px; color: #333; max-width: 1200px; margin: 0 auto; }}
            .header {{ background-color: #f8f9fa; padding: 30px; border-radius: 12px; margin-bottom: 30px; border: 1px solid #e1e4e8; }}
            .summary-stats {{ display: flex; justify-content: space-between; flex-wrap: wrap; gap: 20px; }}
            .stat-box {{ background-color: #fff; border-radius: 8px; padding: 20px; box-shadow: 0 4px 6px rgba(0,0,0,0.05); flex: 1; min-width: 200px; border: 1px solid #eee; }}
            .stat-value {{ font-size: 28px; font-weight: bold; margin-top: 10px; color: #1a73e8; }}
            table {{ width: 100%; border-collapse: collapse; margin-top: 30px; background: #fff; border-radius: 8px; overflow: hidden; box-shadow: 0 4px 6px rgba(0,0,0,0.05); }}
            th {{ background-color: #3c4043; color: white; text-align: left; padding: 15px; font-weight: 500; }}
            td {{ padding: 15px; border-bottom: 1px solid #eee; vertical-align: top; }}
            tr:hover {{ background-color: #fcfcfc; }}
            .score {{ font-weight: bold; padding: 6px 12px; border-radius: 20px; display: inline-block; font-size: 14px; }}
            .score-high {{ background-color: #e6f4ea; color: #137333; }}
            .score-medium {{ background-color: #fef7e0; color: #b06000; }}
            .score-low {{ background-color: #fce8e6; color: #c5221f; }}
            .output pre {{ background-color: #f8f9fa; padding: 12px; border-radius: 6px; border: 1px solid #e1e4e8; font-family: 'Roboto Mono', monospace; font-size: 13px; white-space: pre-wrap; margin: 0; }}
        </style>
    </head>
    <body>
        <div class="header">
            <h1>Prompt Evaluation Report</h1>
            <div class="summary-stats">
                <div class="stat-box"><div>Total Test Cases</div><div class="stat-value">{total_tests}</div></div>
                <div class="stat-box"><div>Average Score</div><div class="stat-value">{avg_score:.1f} / {max_possible_score}</div></div>
                <div class="stat-box"><div>Pass Rate (≥7)</div><div class="stat-value">{pass_rate:.1f}%</div></div>
            </div>
        </div>
        <table>
            <thead><tr><th>Scenario</th><th>Inputs</th><th>Criteria</th><th>Output</th><th>Score</th><th>Reasoning</th></tr></thead>
            <tbody>
    """

    for result in evaluation_results:
        inputs_html = "<br>".join([f"<b>{k}:</b> {v}" for k, v in result["test_case"]["prompt_inputs"].items()])
        criteria_html = "<ul>" + "".join([f"<li>{c}</li>" for c in result["test_case"]["solution_criteria"]]) + "</ul>"
        
        s = result["score"]
        s_cls = "score-high" if s >= 8 else ("score-low" if s <= 5 else "score-medium")
        
        html += f"""
            <tr>
                <td>{result['test_case']['scenario']}</td>
                <td>{inputs_html}</td>
                <td>{criteria_html}</td>
                <td class='output'><pre>{result['output']}</pre></td>
                <td><span class='score {s_cls}'>{s}</span></td>
                <td>{result['reasoning']}</td>
            </tr>
        """
    html += "</tbody></table></body></html>"
    return html

## 3. Improved `PromptEvaluator` with Advanced Techniques

In this version, we rewrite the internal prompts of the `PromptEvaluator` using **Response Schemas** for guaranteed JSON validity.

In [38]:
class IdeaList(BaseModel):
    ideas: List[str]

class InputField(BaseModel):
    key: str
    value: str

class TestCase(BaseModel):
    prompt_inputs: List[InputField]
    solution_criteria: List[str]

class Evaluation(BaseModel):
    reasoning: str
    strengths: List[str]
    weaknesses: List[str]
    score: int

class ImprovedPromptEvaluator:
    def __init__(self, max_concurrent_tasks=2):
        self.max_concurrent_tasks = max_concurrent_tasks

    def generate_unique_ideas(self, task_description, prompt_inputs_spec, num_cases):
        prompt = f"""
        You are a specialized Quality Assurance Engineer. Brainstorm {num_cases} diverse scenarios to test an AI model.


        <task_description>
        {task_description}
        </task_description>

        <input_spec>
        {prompt_inputs_spec}
        </input_spec>

        Guidelines:
        1. Scenarios should range from 'happy path' to 'extreme edge cases'.
        2. Ensure ideas are distinct.
        """
        system = "You are an expert at finding edge cases. Output JSON matching the provided schema."
        
        response = generate_content_with_retry(
            client=client,
            model=model,
            contents=prompt,
            config=types.GenerateContentConfig(
                system_instruction=system,
                response_mime_type="application/json",
                response_schema=IdeaList,
                max_output_tokens=2048
            ),
        )
        return response.parsed.ideas

    def generate_test_case(self, task_description, idea, prompt_inputs_spec):
        prompt = f"""
        Create a detailed test case for:
        Task: {task_description}
        Scenario: {idea}
        Allowed Input Keys: {list(prompt_inputs_spec.keys())}
        """
        system = "You are a meticulous test engineer. Output JSON matching the provided schema."
        
        response = generate_content_with_retry(
            client=client,
            model=model,
            contents=prompt,
            config=types.GenerateContentConfig(
                system_instruction=system,
                response_mime_type="application/json",
                response_schema=TestCase,
                max_output_tokens=4096
            ),
        )
        parsed_tc = response.parsed
        
        prompt_inputs = {field.key: field.value for field in parsed_tc.prompt_inputs}
        
        tc = {
            "prompt_inputs": prompt_inputs,
            "solution_criteria": parsed_tc.solution_criteria,
            "task_description": task_description,
            "scenario": idea
        }
        return tc

    def grade_output(self, test_case, output, extra_criteria):
        prompt = f"""
        Evaluate this AI response:
        <task>{test_case['task_description']}</task>
        <inputs>{test_case['prompt_inputs']}</inputs>
        <response>{output}</response>
        <criteria>{test_case['solution_criteria']} {extra_criteria}</criteria>
        """
        system = "You are an impartial, strict judge. Output JSON matching the provided schema."
        
        response = generate_content_with_retry(
            client=client,
            model=model,
            contents=prompt,
            config=types.GenerateContentConfig(
                system_instruction=system,
                response_mime_type="application/json",
                response_schema=Evaluation,
                max_output_tokens=4096,
                temperature=0.0
            ),
        )
        return response.parsed.model_dump()

    def generate_dataset(self, task_description, prompt_inputs_spec, num_cases=1, output_file="../../data/dataset_improved.json"):
        ideas = self.generate_unique_ideas(task_description, prompt_inputs_spec, num_cases)
        dataset = []
        with concurrent.futures.ThreadPoolExecutor(max_workers=self.max_concurrent_tasks) as executor:
            futures = [executor.submit(self.generate_test_case, task_description, idea, prompt_inputs_spec) for idea in ideas]
            for f in concurrent.futures.as_completed(futures):
                dataset.append(f.result())
                print(f"Generated {len(dataset)}/{len(ideas)} test cases")
        with open(output_file, "w") as f: json.dump(dataset, f, indent=2)
        return dataset

    def run_evaluation(self, run_prompt_fn, dataset_file, extra_criteria=None, html_file="../../reports/improved_report.html"):
        with open(dataset_file, "r") as f: dataset = json.load(f)
        results = []
        for tc in dataset:
            output = run_prompt_fn(tc["prompt_inputs"])
            grade = self.grade_output(tc, output, extra_criteria)
            results.append({"output": output, "test_case": tc, "score": grade["score"], "reasoning": grade["reasoning"]})
            print(f"Graded {len(results)}/{len(dataset)} cases")
        
        avg = mean([r["score"] for r in results])
        print(f"Average Score: {avg:.2f}")
        
        html = generate_prompt_evaluation_report(results)
        with open(html_file, "w", encoding="utf-8") as f: f.write(html)
        display(HTML(f"<h3>Improved Evaluation Complete!</h3><p>Average Score: {avg:.2f}</p>"))
        return results

## 4. Improved Task Prompt

Now we improve the actual meal-plan prompt we are testing. 

### Changes:
- **Structure:** Used XML tags to separate the athlete info from the instructions.
- **Specificity:** Defined exactly what "meal timing" and "macro breakdown" means.
- **Directness:** Instructed the model to be concise and skip conversational filler.

In [39]:
def run_improved_prompt(inputs):
    prompt = f"""
    You are a professional sports nutritionist.
    Generate a concise one-day meal plan based on the following athlete profile:

    <athlete_profile>
    - Height: {inputs['height']}cm
    - Weight: {inputs['weight']}kg
    - Goal: {inputs['goal']}
    - Restrictions: {inputs['restrictions']}
    </athlete_profile>

    Your output MUST include:
    1. <calories>Total daily calorie count</calories>
    2. <macros>Total Protein, Fat, and Carbs in grams</macros>
    3. <schedule>A list of meals including specific times (e.g. 08:00), food items, and portions in grams.</schedule>

    Be direct. Do not include any introductory or concluding remarks.
    """
    return chat([{"role": "user", "content": prompt}], temperature=0.7)

## 5. Run Improved Evaluation

Finally, we run the evaluation with the new, improved prompts.

In [40]:
evaluator = ImprovedPromptEvaluator(max_concurrent_tasks=1)

print("Generating improved dataset...")
dataset = evaluator.generate_dataset(
    task_description="Write a professional, macro-balanced 1 day meal plan for an athlete",
    prompt_inputs_spec={
        "height": "cm",
        "weight": "kg",
        "goal": "athletic performance goal",
        "restrictions": "dietary restrictions",
    },
    num_cases=2,
    output_file="../../data/dataset_improved.json"
 )

print("\nRunning improved evaluation...")
results = evaluator.run_evaluation(
    run_prompt_fn=run_improved_prompt,
    dataset_file="../../data/dataset_improved.json",
    extra_criteria="The response must be strictly formatted with the requested XML-style tags.",
    html_file="../../reports/improved_report.html"
 )

Generating improved dataset...
Generated 1/2 test cases
Generated 2/2 test cases

Running improved evaluation...
Graded 1/2 cases
Gemini rate limit hit. Retrying in 2.0s (attempt 2/5)
Gemini rate limit hit. Retrying in 2.0s (attempt 3/5)
Gemini rate limit hit. Retrying in 4.0s (attempt 4/5)
Gemini rate limit hit. Retrying in 8.0s (attempt 5/5)
Graded 2/2 cases
Average Score: 2.00


In [35]:
# This will print every model ID available to you
for m in client.models.list():
    print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-pro-preview-12-2